In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# AI RISK 의미 분할 시스템 (MIT AI Risk Repository 기반) - REFACTORED
# ═══════════════════════════════════════════════════════════════════════
import os
import pandas as pd
from openai import OpenAI
from tqdm import tqdm
import re
import json
import time
import logging
from datetime import datetime
from typing import List, Dict, Optional
from dataclasses import dataclass, asdict

# ═══════════════════════════════════════════════════════════════════════
# 1. 로깅 설정
# ═══════════════════════════════════════════════════════════════════════
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("ai_risk_semantic_chunking.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# ═══════════════════════════════════════════════════════════════════════
# 2. 데이터 클래스 정의
# ═══════════════════════════════════════════════════════════════════════
@dataclass
class AIRiskChunk:
    """AI 리스크 청크 데이터 클래스"""
    chunk: str
    reason: str
    keywords: List[str]

@dataclass
class AIRiskAnalysisResult:
    """AI 리스크 분석 결과 데이터 클래스"""
    negative_chunks: List[AIRiskChunk]
    has_negative_mention: bool
    parse_error: bool = False

# ═══════════════════════════════════════════════════════════════════════
# 3. MIT AI Risk Repository 기반 프롬프트 (통합된 단일 버전)
# ═══════════════════════════════════════════════════════════════════════
SYSTEM_PROMPT = """당신은 인공지능 위험 분석 전문가이다.

**AI 위험 정의:** AI의 개발, 배포, 사용으로 인해 발생하는 피해의 확률과 심각성의 조합

**7가지 AI 위험 영역:**
1. **차별 및 유해성 (Discrimination & Toxicity)**: AI가 특정 집단에 대한 체계적 불이익을 조장하거나 차별적 의사결정을 내리는 위험
2. **프라이버시 및 보안 (Privacy & Security)**: AI가 사용자의 동의 없이 개인정보를 수집·저장·유출하거나 추론하는 위험
3. **오정보 (Misinformation)**: AI가 사실과 다르거나 오해의 소지가 있는 정보를 의도치 않게 생성하거나 확산
4. **악의적 행위자의 오용 (Malicious Actors & Misuse)**: 악의적 행위자가 AI를 대규모 허위정보 유포, 감시, 사이버 공격, 무기 개발에 활용
5. **인간-컴퓨터 상호작용 (Human-Computer Interaction)**: 인간이 AI에 과도하게 의존하거나 AI를 의인화하여 부적절한 관계 형성
6. **사회경제적 및 환경적 피해 (Socioeconomic & Environmental Harm)**: AI로 인한 일자리 자동화로 고용 질 저하, 불평등 심화, 환경 파괴
7. **AI 시스템 안전, 실패 및 한계 (AI System Safety, Failures, & Limitations)**: AI 시스템이 인간의 목표나 가치와 충돌하는 방식으로 행동

반드시 JSON 형식으로만 출력하라."""

USER_PROMPT_TEMPLATE = """다음 텍스트에서 인공지능 위험을 식별하라.

**[AI 위험의 정의]**
AI 위험은 AI의 개발, 배포, 사용으로 인해 발생하는 피해의 확률과 심각성의 조합이다.
구체적으로는 인공지능 및 인공지능을 활용·개발하는 인간에 의해 발생할 수 있는 부정적인 영향이나 잠재적인 문제점을 의미한다.

**[7가지 주요 영역별 상세 설명]**

**1. 차별 및 유해성 (Discrimination & Toxicity)**
- 편향된 알고리즘, 혐오·폭력·차별적 콘텐츠 생성
- 특정 그룹의 의사결정·인정·생활·종교를 향한 차별적 정보 생성
- 예시: "AI 채용 시스템이 여성 지원자를 차별", "AI가 인종차별적 콘텐츠 생성"

**2. 프라이버시 및 보안 (Privacy & Security)**
- AI 시스템 보안 취약점으로 인한 개인정보 침해, 시스템 조작, 해킹
- 예시: "AI가 개인 위치정보 무단 수집", "AI 시스템 해킹으로 데이터 유출"

**3. 오정보 (Misinformation)**
- 필터버블, 현실 인식의 왜곡, 정보 생태계 오염, 사회적 결속 약화
- 예시: "AI가 거짓 의학정보 생성", "AI 추천 알고리즘이 필터버블 심화"

**4. 악의적 행위자의 오용 (Malicious Actors & Misuse)**
- 사기, 해킹, 표적 조작 등 개인적 이득을 위한 AI 오용
- 예시: "AI 딥페이크로 대규모 정치 조작", "AI 활용 사이버 무기 개발"

**5. 인간-컴퓨터 상호작용 (Human-Computer Interaction)**
- AI에 중요 결정을 과도하게 위임하여 인간의 주체성·자율성 저하
- 예시: "AI 의존으로 인간의 판단력 저하", "AI에 과도한 신뢰로 안전 사고"

**6. 사회경제적 및 환경적 피해 (Socioeconomic & Environmental Harm)**
- 권력 집중, 불평등 심화, 경쟁 역학 변화, 거버넌스 실패
- AI 작동·학습에 요구되는 대규모 자원 소비로 환경 파괴
- 예시: "AI 도입으로 대량 실직 발생", "AI 학습에 막대한 전력 소비로 환경 파괴"

**7. AI 시스템 안전, 실패 및 한계 (AI System Safety, Failures, & Limitations)**
- AI의 기만, 통제 불가능, 해석 불가능 등 위험한 능력
- 능력이나 견고성 부족, 투명성·해석가능성 부족으로 인한 실패
- 예시: "AI 자율주행차 오작동으로 사고", "AI 의료 진단 오류로 환자 피해"

**[AI 위험 인과 분류]**
- **주체(Entity)**: 인간에 의한 위험 / AI에 의한 위험 / 기타
- **의도성(Intentionality)**: 의도적 위험 / 의도하지 않은 위험
- **시점(Timing)**: 배포 전 위험 / 배포 후 위험

**[텍스트]**
{text}

**[중요한 추출 규칙]**

✅ **추출 대상 (명확한 AI 위험 언급):**
- AI 시스템의 오작동, 편향성, 프라이버시 침해 등 명시적 위험
- 전문가의 경고, 우려, 문제 제기
- AI로 인한 일자리 감소, 딥페이크 위험, 허위정보 확산 등

❌ **추출 금지 (AI 위험이 아님):**
- 단순 AI 사용/도입/적용 사실
- AI가 문제를 해결하는 경우
- AI 도입 전 기존 문제
- 추상적이거나 모호한 언급

**[의미 분할 원칙]**
1. 같은 주제의 AI 위험은 하나의 청크로 통합
2. 새로운 종류의 AI 위험이 등장할 때만 분할
3. 원문을 수정 없이 그대로 인용

**[출력 형식]**
{{
    "negative_chunks": [
        {{
            "chunk": "원문 그대로 인용",
            "reason": "7가지 영역 중 해당하는 위험 설명",
            "keywords": ["키워드1", "키워드2", "키워드3"]
        }}
    ],
    "has_negative_mention": true/false
}}

JSON만 출력:"""

# ═══════════════════════════════════════════════════════════════════════
# 4. 텍스트 처리 유틸리티
# ═══════════════════════════════════════════════════════════════════════
class TextProcessor:
    """텍스트 처리 유틸리티 클래스"""

    @staticmethod
    def clean_text(text: str) -> str:
        """텍스트 정제"""
        if pd.isna(text) or text == "":
            return ""

        text = str(text)
        # 제어 문자 제거
        text = re.sub(r'[\x00-\x1f\x7f-\x9f]', ' ', text)
        # 공백 정규화
        text = re.sub(r'\s+', ' ', text)
        return text.strip()

    @staticmethod
    def parse_json_response(response: str) -> Dict:
        """JSON 응답 파싱"""
        try:
            # 텍스트 정제
            response = TextProcessor.clean_text(response)

            # 마크다운 코드 블록 제거
            response = re.sub(r'```json\s*', '', response)
            response = re.sub(r'```\s*', '', response)

            # JSON 추출
            match = re.search(r'\{.*\}', response, re.DOTALL)
            if match:
                result = json.loads(match.group(0))

                # 필수 필드 검증 및 기본값 설정
                if "negative_chunks" not in result:
                    result["negative_chunks"] = []
                if "has_negative_mention" not in result:
                    result["has_negative_mention"] = len(result["negative_chunks"]) > 0

                return result

            return {"negative_chunks": [], "has_negative_mention": False, "parse_error": True}

        except json.JSONDecodeError as e:
            logger.warning(f"JSON 파싱 실패: {e}")
            return {"negative_chunks": [], "has_negative_mention": False, "parse_error": True}
        except Exception as e:
            logger.error(f"예외 발생: {e}")
            return {"negative_chunks": [], "has_negative_mention": False, "parse_error": True}

# ═══════════════════════════════════════════════════════════════════════
# 5. 체크포인트 관리자
# ═══════════════════════════════════════════════════════════════════════
class CheckpointManager:
    """체크포인트 관리 클래스"""

    def __init__(self, output_file: str):
        base_name = os.path.splitext(output_file)[0]
        self.checkpoint_path = f"{base_name}_checkpoint.csv"
        self.index_file_path = f"{base_name}_processed_indices.txt"

    def save_checkpoint(self, results: List[Dict], columns: List[str]):
        """체크포인트 저장"""
        if results:
            result_df = pd.DataFrame(results)
            analysis_cols = ["chunk", "reason", "keywords", "processed_at", "original_index"]
            all_cols = columns + [col for col in analysis_cols if col not in columns]
            existing_cols = [col for col in all_cols if col in result_df.columns]
            result_df = result_df[existing_cols]
            result_df.to_csv(self.checkpoint_path, index=False, encoding='utf-8-sig')
        else:
            empty_df = pd.DataFrame(columns=columns + ["chunk", "reason", "keywords", "processed_at"])
            empty_df.to_csv(self.checkpoint_path, index=False, encoding='utf-8-sig')

    def load_checkpoint_results(self) -> List[Dict]:
        """체크포인트 결과 로드"""
        if os.path.exists(self.checkpoint_path):
            try:
                df = pd.read_csv(self.checkpoint_path, encoding='utf-8-sig')
                if len(df) > 0:
                    logger.info(f"📂 체크포인트 로드: {len(df)}개 행")
                    return df.to_dict('records')
            except Exception as e:
                logger.error(f"체크포인트 로드 실패: {e}")
        return []

    def save_processed_indices(self, indices: List[int]):
        """처리된 인덱스 저장"""
        with open(self.index_file_path, 'w') as f:
            for idx in indices:
                f.write(f"{idx}\n")

    def load_processed_indices(self) -> List[int]:
        """처리된 인덱스 로드"""
        if os.path.exists(self.index_file_path):
            with open(self.index_file_path, 'r') as f:
                return [int(line.strip()) for line in f if line.strip()]
        return []

# ═══════════════════════════════════════════════════════════════════════
# 6. AI RISK 의미 분할 분석기 (핵심 클래스)
# ═══════════════════════════════════════════════════════════════════════
class AIRiskSemanticChunker:
    """MIT AI Risk Repository 기반 의미 분할 분석기"""

    def __init__(self, api_base: str = "http://221.155.102.177:1234/v1",
                 model_name: str = "lgai-exaone_exaone-4.0.1-32b"):
        """
        AI Risk 의미 분할 분석기 초기화

        Args:
            api_base: LM Studio API 엔드포인트
            model_name: 사용할 모델 이름
        """
        self.client = OpenAI(base_url=api_base, api_key="not-needed")
        self.model_name = model_name
        self.text_processor = TextProcessor()

        logger.info(f"🚀 AI RISK 의미 분할 분석기 초기화 (MIT Repository 기반)")
        logger.info(f"API: {api_base}")
        logger.info(f"모델: {model_name}")

        # 연결 테스트
        self._test_connection()

    def _test_connection(self):
        """API 연결 테스트"""
        try:
            response = self.client.chat.completions.create(
                model=self.model_name,
                messages=[{"role": "user", "content": "안녕하세요"}],
                max_tokens=10
            )
            logger.info("✅ 연결 성공")
        except Exception as e:
            logger.error(f"❌ 연결 실패: {e}")
            raise

    def analyze_single_text(self, text: str, max_retries: int = 2) -> Dict:
        """
        단일 텍스트 AI Risk 분석

        Args:
            text: 분석할 텍스트
            max_retries: 최대 재시도 횟수

        Returns:
            분석 결과 딕셔너리
        """
        # 텍스트 정제
        cleaned_text = self.text_processor.clean_text(text)
        if not cleaned_text:
            return {
                "negative_chunks": [],
                "has_negative_mention": False,
                "parse_error": False
            }

        # 프롬프트 생성
        user_prompt = USER_PROMPT_TEMPLATE.format(text=cleaned_text)

        for attempt in range(max_retries):
            try:
                response = self.client.chat.completions.create(
                    model=self.model_name,
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": user_prompt}
                    ],
                    temperature=0.1,
                    max_tokens=5000,
                )

                # 응답 파싱
                result = self.text_processor.parse_json_response(
                    response.choices[0].message.content
                )

                if not result.get("parse_error", False):
                    return result

            except Exception as e:
                logger.warning(f"시도 {attempt + 1}/{max_retries} 실패: {e}")
                if attempt < max_retries - 1:
                    time.sleep(1)

        return {
            "negative_chunks": [],
            "has_negative_mention": False,
            "parse_error": True
        }

    def process_csv(self,
                   input_file: str,
                   output_file: str,
                   text_column: str = 'text',
                   checkpoint_interval: int = 10) -> List[Dict]:
        """
        CSV 파일 처리 (체크포인트 지원)

        Args:
            input_file: 입력 CSV 파일 경로
            output_file: 출력 CSV 파일 경로
            text_column: 텍스트 컬럼명
            checkpoint_interval: 체크포인트 저장 간격

        Returns:
            분석 결과 리스트
        """
        # CSV 로드
        logger.info(f"📚 CSV 파일 로드: {input_file}")
        df = pd.read_csv(input_file, encoding='utf-8-sig')

        if text_column not in df.columns:
            raise ValueError(f"❌ '{text_column}' 컬럼이 없습니다")

        # 체크포인트 관리자 초기화
        checkpoint_manager = CheckpointManager(output_file)

        # 기존 진행상황 로드
        processed_indices = checkpoint_manager.load_processed_indices()
        all_results = checkpoint_manager.load_checkpoint_results()
        processed_set = set(processed_indices)

        logger.info(f"📊 총 {len(df)}개 행 (이미 처리: {len(processed_set)}개)")

        # 진행 상황 표시
        pbar = tqdm(df.iterrows(), total=len(df), desc="🔍 AI RISK 분석 중")

        for idx, row in pbar:
            # 이미 처리된 행 스킵
            if idx in processed_set:
                continue

            text = row[text_column]

            # 빈 텍스트 스킵
            if pd.isna(text) or str(text).strip() == "":
                processed_indices.append(idx)
                continue

            # AI Risk 분석
            result = self.analyze_single_text(text)

            # 결과 저장 (AI Risk가 있는 경우만)
            if not result.get("parse_error", False) and result.get("has_negative_mention", False):
                for chunk_info in result.get("negative_chunks", []):
                    result_row = self._create_result_row(row, idx, chunk_info)
                    all_results.append(result_row)

            processed_indices.append(idx)

            # 주기적 체크포인트 저장
            if len(processed_indices) % checkpoint_interval == 0:
                checkpoint_manager.save_checkpoint(all_results, list(df.columns))
                checkpoint_manager.save_processed_indices(processed_indices)

            # 진행률 업데이트
            pbar.set_postfix({
                'processed': len(processed_indices),
                'ai_risks': len(all_results)
            })

            time.sleep(0.1)  # API 부하 방지

        pbar.close()

        # 최종 결과 저장
        self._save_final_results(df, all_results, output_file)

        # 최종 체크포인트 저장
        checkpoint_manager.save_checkpoint(all_results, list(df.columns))
        checkpoint_manager.save_processed_indices(processed_indices)

        return all_results

    def _create_result_row(self, row: pd.Series, idx: int, chunk_info: Dict) -> Dict:
        """결과 행 생성"""
        result_row = row.to_dict()
        result_row['original_index'] = idx

        # 키워드 처리
        keywords = chunk_info.get("keywords", [])
        keywords_str = ", ".join(keywords) if isinstance(keywords, list) else str(keywords)

        result_row.update({
            "chunk": chunk_info.get("chunk", ""),
            "reason": chunk_info.get("reason", ""),
            "keywords": keywords_str,
            "processed_at": datetime.now().isoformat()
        })

        return result_row

    def _save_final_results(self, df: pd.DataFrame, results: List[Dict], output_file: str):
        """최종 결과 저장"""
        logger.info(f"\n{'='*60}")
        logger.info(f"🎉 AI RISK 분석 완료!")
        logger.info(f"{'='*60}")
        logger.info(f"📊 총 처리: {len(df)}개")
        logger.info(f"🚨 발견된 AI RISK: {len(results)}개")

        if results:
            result_df = pd.DataFrame(results)

            # 컬럼 순서 정리
            analysis_columns = ["chunk", "reason", "keywords", "processed_at", "original_index"]
            original_columns = [col for col in df.columns if col in result_df.columns]
            ordered_columns = original_columns + [col for col in analysis_columns if col in result_df.columns]

            result_df = result_df[ordered_columns]
            result_df.to_csv(output_file, index=False, encoding='utf-8-sig')
            logger.info(f"💾 결과 저장: {output_file}")
        else:
            logger.warning(f"⚠️ AI RISK가 발견되지 않았습니다")
            empty_df = pd.DataFrame(columns=list(df.columns) + ["chunk", "reason", "keywords", "processed_at"])
            empty_df.to_csv(output_file, index=False, encoding='utf-8-sig')

# ═══════════════════════════════════════════════════════════════════════
# 7. 메인 함수
# ═══════════════════════════════════════════════════════════════════════
def main():
    """메인 실행 함수"""
    logger.info("="*60)
    logger.info("🚀 AI RISK 의미 분할 시스템 시작 (MIT Repository)")
    logger.info("="*60 + "\n")

    # ═══════════════════════════════════════════════════════════════════
    # 설정값
    # ═══════════════════════════════════════════════════════════════════

    # 파일 경로
    INPUT_FILE = "/mnt/c/Users/com/Desktop/논문작업/2025_AI위험/0. Data/2. preprocesseddata/Ai_전처리데이터2.csv"
    OUTPUT_FILE = "/mnt/c/Users/com/Desktop/논문작업/2025_AI위험/7. 투고/2.의미분할/SC_MIT_FINAL_20251023.csv"

    # 기타 설정
    TEXT_COLUMN = "text"
    API_BASE = "http://221.155.102.177:1234/v1"
    MODEL_NAME = "lgai-exaone_exaone-4.0.1-32b"
    CHECKPOINT_INTERVAL = 10

    # ═══════════════════════════════════════════════════════════════════

    # 경로 검증
    if not os.path.exists(INPUT_FILE):
        logger.error(f"❌ 입력 파일 없음: {INPUT_FILE}")
        return None

    # 출력 디렉토리 생성
    output_dir = os.path.dirname(OUTPUT_FILE)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)
        logger.info(f"📁 출력 디렉토리 생성: {output_dir}")

    # 타임스탬프 추가
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    base_name, ext = os.path.splitext(OUTPUT_FILE)
    output_path = f"{base_name}_{timestamp}{ext}"

    logger.info(f"🎯 분석 대상: AI 위험 (MIT AI Risk Repository 기반)")
    logger.info(f"📄 입력: {INPUT_FILE}")
    logger.info(f"📄 출력: {output_path}")
    logger.info(f"🔧 API: {API_BASE}")
    logger.info(f"🤖 모델: {MODEL_NAME}\n")

    try:
        # 분석 실행
        chunker = AIRiskSemanticChunker(
            api_base=API_BASE,
            model_name=MODEL_NAME
        )

        results = chunker.process_csv(
            input_file=INPUT_FILE,
            output_file=output_path,
            text_column=TEXT_COLUMN,
            checkpoint_interval=CHECKPOINT_INTERVAL
        )

        logger.info("🎊 모든 작업 완료!")
        return results

    except Exception as e:
        logger.error(f"❌ 처리 중 오류: {e}")
        raise

if __name__ == "__main__":
    main()